# 03 - Evaluation

Runs the 30-question golden set through both models. Retrieval metrics (Context Precision/Recall, MRR) come from the gold pages; generation metrics (Faithfulness, Relevancy, Correctness) come from an LLM judge (gemma3:4b, a different model from both generators).

In [1]:
# %pip install -q ollama chromadb pandas

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [3]:
from src.evaluation import run_evaluation

smol_rows = run_evaluation("smollm")
llama_rows = run_evaluation("llama")

Q01 done
Q02 done
Q03 done
Q04 done
Q05 done
Q06 done
Q07 done
Q08 done
Q09 done
Q10 done
Q11 done
Q12 done
Q13 done
Q14 done
Q15 done
Q16 done
Q17 done
Q18 done
Q19 done
Q20 done
Q21 done
Q22 done
Q23 done
Q24 done
Q25 done
Q26 done
Q27 done
Q28 done
Q29 done
Q30 done
smollm summary: {'context_precision': 0.25, 'context_recall': 0.923, 'mrr': 0.821, 'faithfulness': 0.767, 'relevancy': 0.8, 'correctness': 0.467, 'latency_seconds': 0.471}
Q01 done
Q02 done
Q03 done
Q04 done
Q05 done
Q06 done
Q07 done
Q08 done
Q09 done
Q10 done
Q11 done
Q12 done
Q13 done
Q14 done
Q15 done
Q16 done
Q17 done
Q18 done
Q19 done
Q20 done
Q21 done
Q22 done
Q23 done
Q24 done
Q25 done
Q26 done
Q27 done
Q28 done
Q29 done
Q30 done
llama summary: {'context_precision': 0.25, 'context_recall': 0.923, 'mrr': 0.821, 'faithfulness': 0.833, 'relevancy': 0.8, 'correctness': 0.9, 'latency_seconds': 12.062}


### Compare the two models

In [4]:
import json, pandas as pd
from src.config import RESULTS_DIR

smol_summary = json.loads((RESULTS_DIR / "smollm_summary.json").read_text(encoding="utf-8"))
llama_summary = json.loads((RESULTS_DIR / "llama_summary.json").read_text(encoding="utf-8"))
pd.DataFrame({"smollm2-135m": smol_summary, "llama3.2-3b": llama_summary})

,smollm2-135m,llama3.2-3b
context_precision,0.250,0.250
context_recall,0.923,0.923
mrr,0.821,0.821
faithfulness,0.767,0.833
relevancy,0.800,0.800
correctness,0.467,0.900
latency_seconds,0.471,12.062


### How many questions each model got right

In [5]:
correct_counts = {}

for model_name, rows in [("SmolLM2-135M", smol_rows), ("Llama 3.2-3B", llama_rows)]:
    full = 0
    half = 0
    wrong = 0
    for r in rows:
        if r["correctness"] == 1.0:
            full += 1
        elif r["correctness"] == 0.5:
            half += 1
        else:
            wrong += 1
    correct_counts[model_name] = {"Fully correct (1.0)": full, "Partially correct (0.5)": half, "Wrong (0.0)": wrong, "Out of": len(rows)}

pd.DataFrame(correct_counts).T

,Fully correct (1.0),Partially correct (0.5),Wrong (0.0),Out of
SmolLM2-135M,14,0,16,30
Llama 3.2-3B,27,0,3,30
